In [ ]:
!git clone https://github.com/sciai-lab/Truth_is_Universal.git

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

from huggingface_hub import login

In [ ]:
RAW_DATA_DIR = Path("/content/Truth_is_Universal/datasets")

FILE_GLOB = "*.csv"

TEXT_COL = "statement"
LABEL_COL = "label"


TRAIN_FRAC = 0.7
VAL_FRAC = 0.15
TEST_FRAC = 0.15

In [ ]:
# Directories to skip entirely (e.g., real-world lie scenarios)
EXCLUDE_DIRS = {"real_world_scenarios"}

# Manual metadata overrides for strange file names
# stems = filename without extension
MANUAL_METADATA = {
    # Diverse test sets (used as eval-only in TiU)
    "common_claim_true_false": {
        "topic": "common_claim",
        "polarity": "affirmed",
        "statement_type": "simple",
        "language": "en",
        "role": "eval_only",
    },
    "counterfact_true_false": {
        "topic": "counterfact",
        "polarity": "affirmed",
        "statement_type": "simple",
        "language": "en",
        "role": "eval_only",
    },
}


In [ ]:
def iter_tiu_files(root: Path):
    """Yield all CSVs under root, excluding EXCLUDE_DIRS."""
    for path in root.rglob("*.csv"):
        # Skip anything under excluded subdirs
        if any(part in EXCLUDE_DIRS for part in path.parts):
            continue
        yield path


In [ ]:
def parse_dataset_name(path: Path):
    """
    Infer topic / polarity / statement_type / language from filename.
    Uses MANUAL_METADATA first, then a generic rule for the regular TiU files.
    """
    stem = path.stem
    lower = stem.lower()

    if lower in MANUAL_METADATA:
        meta = MANUAL_METADATA[lower].copy()
        meta.setdefault("source_dataset", lower)
        return meta

    original_name = lower

    language = "en"
    if lower.endswith("_de"):
        language = "de"
        lower = lower[:-3]  # strip "_de"

    # Statement type
    statement_type = "simple"
    if lower.endswith("_conj"):
        statement_type = "conjunction"
        lower = lower[:-5]  # strip "_conj"
    elif lower.endswith("_disj"):
        statement_type = "disjunction"
        lower = lower[:-5]  # strip "_disj"

    # Polarity
    polarity = "affirmed"
    if lower.startswith("neg_"):
        polarity = "negated"
        lower = lower[4:]  # strip "neg_"

    topic = lower  # remaining piece

    meta = {
        "topic": topic,
        "polarity": polarity,
        "statement_type": statement_type,
        "language": language,
        "source_dataset": original_name,
    }
    return meta

In [ ]:
def load_one_table(path: Path) -> pd.DataFrame:
    """Load CSV and attach standard metadata columns."""
    df = pd.read_csv(path)

    if TEXT_COL not in df.columns or LABEL_COL not in df.columns:
        raise ValueError(f"{path} is missing {TEXT_COL} or {LABEL_COL} columns")

    df = df.rename(columns={TEXT_COL: "text", LABEL_COL: "label"})

    meta = parse_dataset_name(path)
    for k, v in meta.items():
        df[k] = v

    return df


In [ ]:
dfs = []
for f in iter_tiu_files(RAW_DATA_DIR):
    try:
        df = load_one_table(f)
        dfs.append(df)
    except Exception as e:
        print(f"WARNING: skipping {f} due to error: {e}")

full_df = pd.concat(dfs, ignore_index=True)

# Add stable, per‑example ID: <source_dataset>:<0-based index within that file>
# 'source_dataset' is already set in parse_dataset_name().
full_df["example_id"] = (
    full_df["source_dataset"].astype(str)
    + ":"
    + full_df.groupby("source_dataset").cumcount().astype(str)
)

print("Total examples:", len(full_df))
print("By topic:\n", full_df["topic"].value_counts())
print(
    "By role (if present):\n",
    full_df.get("role", pd.Series(["train_eval"] * len(full_df))).value_counts(),
)


# Mark role for everything without an explicit role as "train_eval"
if "role" not in full_df.columns:
    full_df["role"] = "train_eval"
else:
    full_df["role"] = full_df["role"].fillna("train_eval")

In [ ]:
train_eval_df = full_df[full_df["role"] == "train_eval"].copy()

train_eval_df["stratum"] = train_eval_df["topic"] + "|" + train_eval_df["polarity"]

temp_frac = 1.0 - TRAIN_FRAC
train_df, temp_df = train_test_split(
    train_eval_df,
    test_size=temp_frac,
    random_state=42,
    stratify=train_eval_df["stratum"],
)

relative_test_size = TEST_FRAC / (VAL_FRAC + TEST_FRAC)
val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_size,
    random_state=43,
    stratify=temp_df["stratum"],
)

for df in (train_df, val_df, test_df):
    df.drop(columns=["stratum"], inplace=True)

print(len(train_df), len(val_df), len(test_df))

# Keep the eval-only datasets around too, but separate
eval_only_df = full_df[full_df["role"] == "eval_only"].copy()

# Optional: wrap in HF DatasetDict
from datasets import Dataset, DatasetDict

from datasets import Dataset, DatasetDict

core_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

# extra_eval includes extra attributes that don't match the others, so HF will be unfriendly if added together
extra_eval_dataset = Dataset.from_pandas(eval_only_df.reset_index(drop=True))

In [ ]:
train_df

In [ ]:
train_df.isnull().all()

**Note:** There is extra information provided, but we funcitonally only need the following columns:
```
[
    "example_id", "text", "label",
    "topic", "polarity", "statement_type", "language"
]
```
For now, we'll keep them for debugging in the HF dataset and filter these columns our for any future processing.

In [ ]:
repo_id = "KingTechnician/tiu-splits"  # change this



In [ ]:
core_dataset.push_to_hub(
    repo_id=repo_id,
    private=True
)


In [ ]:
extra_repo_id = "KingTechnician/tiu-splits-extra"
extra_eval_dataset.push_to_hub(
    repo_id=extra_repo_id,
    private=True,
)